# FRED ingestion - orchestrator

Iterates over the configured series, calling the worker notebook once per series via `dbutils.notebook.run`, then appends every result to `` `finhive-2026`.logs.ingestionLog ``.

In [0]:
import os

dbutils.widgets.text("job_name", "")
dbutils.widgets.text("config_path", "")

job_name = dbutils.widgets.get("job_name")
config_path = dbutils.widgets.get("config_path")

if not config_path:
    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    repo_root = "/Workspace" + notebook_path.rsplit("/notebooks/", 1)[0]
    config_path = f"{repo_root}/config/data_ingestion/fred.json"

if not job_name:
    raise ValueError("widget 'job_name' is required")

In [0]:
worker_notebook_path = "./worker"

In [0]:
import json

with open(config_path) as f:
    series_config = json.load(f)

if not series_config:
    raise ValueError(f"no series configured in {config_path}")

In [0]:
from datetime import timedelta

from pyspark.sql import functions as F

series_list = [entry["series"] for entry in series_config]

last_updates = {
    row["series"]: (row["last_date"] + timedelta(days=1)).isoformat()
    for row in (
        spark.table("`finhive-2026`.logs.ingestionLog")
        .filter((F.col("status") == True) & F.col("series").isin(series_list))
        .groupBy("series")
        .agg(F.max("lastObservationDate").alias("last_date"))
        .collect()
    )
}

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_worker(entry):
    series = entry["series"]
    start_date = last_updates.get(series)

    raw_result = dbutils.notebook.run(
        worker_notebook_path,
        timeout_seconds=3600,
        arguments={
            "series": series,
            "start_date": start_date or "",
            "job_name": job_name,
        },
    )
    return json.loads(raw_result)

results = []

max_workers = min(len(series_config), 3)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_entry = {
        executor.submit(run_worker, entry): entry for entry in series_config
    }
    for future in as_completed(future_to_entry):
        result = future.result()
        results.append(result)

In [0]:
from datetime import date, datetime

log_rows = [
    (
        r["series"],
        r["source"],
        r["status"],
        datetime.fromisoformat(r["updateAt"]),
        date.fromisoformat(r["lastObservationDate"]) if r["lastObservationDate"] else None,
        r["item_count"],
        r["error"],
        r["job_name"],
    )
    for r in results
]
log_schema = "series string, source string, status boolean, updateAt timestamp, lastObservationDate date, item_count int, error string, job_name string"
log_df = spark.createDataFrame(log_rows, schema=log_schema)
log_df.write.mode("append").saveAsTable("`finhive-2026`.logs.ingestionLog")

In [0]:
failures = [r for r in results if not r["status"]]
if failures:
    failed_series = [r["series"] for r in failures]
    raise RuntimeError(f"{len(failures)} series failed to ingest: {failed_series}")

print(f"ingested {len(results)} series successfully")